In [1]:
import json
import torch
import requests
from transformers import pipeline

# 1. Caricamento Dati
json_url = "https://raw.githubusercontent.com/Profession-AI/progetti-deeplearning/refs/heads/main/Sintesi%20automatica%20di%20cartelle%20cliniche%20di%20un%E2%80%99azienda%20ospedaliera/hospital_records.json"

try:
    response = requests.get(json_url)
    response.raise_for_status()
    data = response.json()
    print(f"Dati caricati: {len(data)} pazienti trovati.")
except Exception as e:
    print(f"Errore nel caricamento: {e}")
    data = []

# 2. Configurazione Modello
# Utilizziamo BART-large-cnn per una sintesi di alta qualità
device = 0 if torch.cuda.is_available() else -1
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=device)

def generate_fluid_summary(patient_data):
    """
    Trasforma i dati del paziente in un paragrafo narrativo e fluido.
    """
    all_episodes = []
    
    for hosp in patient_data.get("hospitalizations", []):
        # Costruiamo una frase naturale per ogni ricovero
        tests = ", ".join(hosp.get('test_results', []))
        sentence = (
            f"The patient was admitted with {hosp.get('diagnosis')}. "
            f"The clinical history shows {hosp.get('anamnesis')}. "
            f"During the stay, tests such as {tests} were performed. "
            f"The final prognosis indicates {hosp.get('prognosis')}."
        )
        all_episodes.append(sentence)
    
    # Uniamo tutti i ricoveri in un unico testo grezzo
    full_narrative = " ".join(all_episodes)
    input_len = len(full_narrative.split())
    
    
    # Generazione con parametri di fluidità (Beam Search)
    summary_output = summarizer(
        full_narrative,
        max_length=int(input_len * 0.8),
        min_length=int(input_len * 0.3),
        num_beams=4,           # Esplora più sequenze per trovare la più fluida
        no_repeat_ngram_size=3, # Evita ripetizioni fastidiose
        early_stopping=True
    )
    
    return summary_output[0]['summary_text']

# 3. Elaborazione e Salvataggio
if data:
    final_results = []
    
    print("Inizio generazione riassunti (potrebbe richiedere un minuto)...")
    for patient in data:
        name = patient.get("patient_name")
        print(f"Elaborazione: {name}...")
        
        summary = generate_fluid_summary(patient)
        
        final_results.append({
            "patient_name": name,
            "clinical_summary": summary
        })

    # Salvataggio su file JSON
    output_file = "patient_summaries.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(final_results, f, indent=4, ensure_ascii=False)
    
    print(f"\n✅ Operazione completata! File '{output_file}' salvato correttamente.")

# 4. Anteprima Risultato
if final_results:
    print("\n--- Anteprima del primo riassunto ---")
    print(f"Paziente: {final_results[0]['patient_name']}")
    print(f"Riassunto: {final_results[0]['clinical_summary']}")

c:\Users\feder\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Dati caricati: 10 pazienti trovati.



Device set to use cpu


Inizio generazione riassunti (potrebbe richiedere un minuto)...
Elaborazione: Patient 1...
Elaborazione: Patient 2...
Elaborazione: Patient 3...
Elaborazione: Patient 4...
Elaborazione: Patient 5...
Elaborazione: Patient 6...
Elaborazione: Patient 7...
Elaborazione: Patient 8...
Elaborazione: Patient 9...
Elaborazione: Patient 10...

✅ Operazione completata! File 'patient_summaries.json' salvato correttamente.

--- Anteprima del primo riassunto ---
Paziente: Patient 1
Riassunto: The patient was admitted with Bronchitis. The clinical history shows No prior conditions. During the stay, tests such as MRI indicates tissue damage were performed. The final prognosis indicates Surgery recommended.
